<a href="https://colab.research.google.com/github/neowork77/BU-Dorms-RAG-CHAT/blob/main/prepare_data_%2B_embeddings.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install faiss-cpu sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 37.4 MB/s eta 0:00:00


# 🌟 **Prepare Data**

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv(
    "bangkok_university_dorms.csv"
)

def prepare_data(row):

    # NAME
    name_parts = []

    # project name
    if pd.notna(row["project_name"]):
        project_name = str(row["project_name"]).strip()

        if project_name != "":
            name_parts.append(project_name)

    # building
    if pd.notna(row["building"]):

        building = str(row["building"]).strip()

        if building not in ["", "-", "nan"]:
            name_parts.append(f"ตึก {building}")

    # floor
    if pd.notna(row["floor"]):

        floor = str(row["floor"]).strip()

        if floor not in ["", "-", "nan"]:
            name_parts.append(f"ชั้น {floor}")

    name = " ".join(name_parts)

    # PRICE
    if (
        pd.isna(row["price"]) or
        str(row["price"]).strip() in ["", "-", "nan"]
    ):

        price = "ไม่ระบุราคา"

    else:

        try:

            price = (
                f"{float(row['price']):,.0f} "
                f"บาทต่อเดือน"
            )

        except:

            price = str(row["price"])

    # DETAILS
    detail_parts = []

    # room type
    if pd.notna(row["room_type"]):

        room_type = str(row["room_type"]).strip()

        if room_type not in ["", "-", "nan"]:
            detail_parts.append(
                f"ประเภทห้อง {room_type}"
            )

    # room size
    if pd.notna(row["room_size_sqm"]):

        room_size = str(row["room_size_sqm"]).strip()

        if room_size not in ["", "-", "nan"]:
            detail_parts.append(
                f"ขนาด {room_size} ตร.ม."
            )

    # room facilities
    if pd.notna(row["room_facilities"]):

        room_fac = str(
            row["room_facilities"]
        ).strip()

        if room_fac not in ["", "-", "nan"]:

            detail_parts.append(
                f"สิ่งอำนวยความสะดวกภายในห้อง: "
                f"{room_fac}"
            )

    # project facilities
    if pd.notna(row["project_facilities"]):

        project_fac = str(
            row["project_facilities"]
        ).strip()

        if project_fac not in ["", "-", "nan"]:

            detail_parts.append(
                f"สิ่งอำนวยความสะดวกภายในโครงการ: "
                f"{project_fac}"
            )

    details = " | ".join(detail_parts)

    # IMAGE CHECK
    def check_img(img_url):

        if pd.isna(img_url):
            return "ไม่มีรูปภาพ"

        img_url = str(img_url).strip()

        if img_url in ["", "-", "nan"]:
            return "ไม่มีรูปภาพ"

        return img_url

    # RETURN
    return pd.Series([

        name,
        price,
        details,
        row["url"],
        row["latitude"],
        row["longitude"],
        check_img(row["image_1"]),
        check_img(row["image_2"]),
        check_img(row["image_3"]),
        check_img(row["image_4"])
    ])

# RUN
new_cols = [

    "name",
    "price",
    "details",
    "url",
    "lat",
    "lng",
    "img1",
    "img2",
    "img3",
    "img4"
]

df[new_cols] = df.apply(
    prepare_data,
    axis=1
)

# FINAL DATAFRAME
df_final = df[new_cols].copy()

# EXPORT
df_final.to_csv(
    "dorms_final_delivery.csv",
    index=False,
    encoding="utf-8-sig"
)

print("จัดการข้อมูลเรียบร้อย!")

df_final.head()

จัดการข้อมูลเรียบร้อย!


,name,price,details,url,lat,lng,img1,img2,img3,img4
0,dcondo Shine Rangsit (ดีคอนโด ชายน์ รังสิต) ชั...,"11,000 บาทต่อเดือน",ประเภทห้อง 1 ห้องนอน | ขนาด 26.0 ตร.ม. | สิ่งอ...,https://propertyhub.in.th/listings/26-ตร-ม-11-...,14.062188,100.608401,https://bcdn.propertyhub.in.th/pictures/202603...,https://bcdn.propertyhub.in.th/pictures/202603...,https://bcdn.propertyhub.in.th/pictures/202603...,https://bcdn.propertyhub.in.th/pictures/202603...
1,dcondo Campus Resort Rangsit (ดีคอนโด แคมปัส ร...,"8,000 บาทต่อเดือน",ประเภทห้อง สตูดิโอ | ขนาด 29.0 ตร.ม. | สิ่งอำน...,https://propertyhub.in.th/listings/rb23922-ให้...,14.063015,100.607360,https://bcdn.propertyhub.in.th/pictures/202603...,https://bcdn.propertyhub.in.th/pictures/202603...,https://bcdn.propertyhub.in.th/pictures/202603...,https://bcdn.propertyhub.in.th/pictures/202603...
2,dcondo hype rangsit (ดีคอนโด ไฮป์ รังสิต) ชั้น 3,"9,000 บาทต่อเดือน",ประเภทห้อง 1 ห้องนอน | ขนาด 27.52 ตร.ม. | สิ่ง...,https://propertyhub.in.th/listings/ให้เช่า-d-c...,14.041953,100.616681,https://bcdn.propertyhub.in.th/pictures/202605...,https://bcdn.propertyhub.in.th/pictures/202605...,https://bcdn.propertyhub.in.th/pictures/202605...,https://bcdn.propertyhub.in.th/pictures/202605...
3,Kave Wonderland (เคฟ วันเดอร์แลนด์) ชั้น 3,"13,000 บาทต่อเดือน",ประเภทห้อง 1 ห้องนอน | ขนาด 25.0 ตร.ม. | สิ่งอ...,https://propertyhub.in.th/listings/เช่าคอนโด-เ...,14.065818,100.611212,https://bcdn.propertyhub.in.th/pictures/202605...,https://bcdn.propertyhub.in.th/pictures/202605...,https://bcdn.propertyhub.in.th/pictures/202605...,https://bcdn.propertyhub.in.th/pictures/202605...
4,dcondo vivid Rangsit (ดีคอนโด วิวิด รังสิต) ตึ...,"11,000 บาทต่อเดือน",ประเภทห้อง 1 ห้องนอน | ขนาด 1.0 ตร.ม. | สิ่งอำ...,https://propertyhub.in.th/listings/ให้เช่า-dco...,14.041670,100.616693,https://bcdn.propertyhub.in.th/pictures/202605...,https://bcdn.propertyhub.in.th/pictures/202605...,https://bcdn.propertyhub.in.th/pictures/202605...,https://bcdn.propertyhub.in.th/pictures/202605...


# 🌟 **Embedding**

In [ ]:
import faiss
from sentence_transformers import SentenceTransformer

df_final = pd.read_csv("dorms_final_delivery.csv")

# CREATE TEXT FOR EMBEDDING

df_final["embed_text"] = (

    "ชื่อโครงการ: " + df_final["name"].astype(str) +

    " | ราคา: " + df_final["price"].astype(str) +

    " | รายละเอียด: " + df_final["details"].astype(str)

)

# LOAD MODEL
model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

# EMBEDDING
embeddings = model.encode(df_final["embed_text"].tolist(),show_progress_bar=True)

embeddings = np.array(embeddings).astype("float32")

# NORMALIZE
faiss.normalize_L2(embeddings)

# CREATE INDEX
index = faiss.IndexFlatIP(
    embeddings.shape[1]
)

index.add(embeddings)

# SAVE
faiss.write_index(
    index,
    "dorms_index.faiss"
)

df_final.to_pickle(
    "dorms_metadata.pkl"
)

print("เสร็จสมบูรณ์!")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/30 [00:00<?, ?it/s]

เสร็จสมบูรณ์!
